In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from optical_setup import OpticalSetupSim2, OpticalSetupSim, OpticalSetup
from utils import max_eig_power, max_singular, reshape_to_square
from matplotlib.animation import FuncAnimation
from scipy.spatial import distance_matrix
from scipy.interpolate import interp1d
from encoding import vectorized_basket
from matplotlib.patches import Circle
from pyALP41.consts import DMD_HEIGHT, DMD_WIDTH
from scipy.interpolate import interp1d
from scipy.signal import correlate2d
from IPython.display import HTML
from scipy.ndimage import maximum_filter
from skimage.feature import peak_local_max

from LightPipes import *
from time import sleep

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("tab10")
colors = sns.color_palette('tab10')

# Aux. functions
---

In [ ]:
def crop_center(img,cropx,cropy):
    y,x = img.shape
    startx = x//2-(cropx//2)
    starty = y//2-(cropy//2)    
    return img[starty:starty+cropy,startx:startx+cropx]

def local_autocorrelation(img, shift_x=0, shift_y=0):
    rows, cols = img.shape
    assert shift_y < rows and shift_x < cols, 'Shift is bigger than the image dimension.'

    # Dimension of rows = vertical
    shifted_img = np.vstack((img, np.zeros((np.abs(shift_y), cols)))) if shift_y < 0 else np.vstack((np.zeros((shift_y, cols)), img))
    shifted_img = shifted_img[-rows:, :] if shift_y < 0 else shifted_img[0:rows, :]

    # Dimension of cols = horizontal
    shifted_img = np.hstack((shifted_img, np.zeros((rows, np.abs(shift_x))))) if shift_x < 0 else np.hstack((np.zeros((rows, shift_x)), shifted_img))
    shifted_img = shifted_img[:, -cols:] if shift_x < 0 else shifted_img[:, 0:cols]

    # Local autocorrelation
    r_xx = np.sum(img * shifted_img)

    return r_xx

def compute_fwhm(x, y):
    y = y / np.max(y)
    half_max = 0.5

    above = y >= half_max
    if not np.any(above):
        return None

    indices = np.where(above)[0]
    left_idx = indices[0] - 1
    right_idx = indices[-1] + 1

    # Interpolation
    f = interp1d(x, y, kind='linear', bounds_error=False, fill_value='extrapolate')
    x_fine = np.linspace(x[left_idx], x[right_idx], 1000)
    y_fine = f(x_fine)

    cross_points = x_fine[np.where(np.diff(np.sign(y_fine - half_max)) != 0)]

    if len(cross_points) >= 2:
        return np.abs(cross_points[-1] - cross_points[0])
    else:
        return None

In [ ]:
def speckle_grain_mean(
    image,
    radius=5,
    min_distance=5,
    threshold_rel=0.5,
    num_peaks=None,
    return_coords=False
):
    """
    Detecta centros de grãos em uma imagem speckle e calcula a média
    dos valores ao redor de cada centro dentro de um raio especificado.

    Parameters
    ----------
    image : 2D numpy array
        Imagem grayscale (speckle)
    radius : int
        Raio da vizinhança (em pixels)
    min_distance : int
        Distância mínima entre centros detectados
    threshold_rel : float
        Threshold relativo (0–1) para detecção de picos
    num_peaks : int or None
        Número máximo de grãos a detectar
    return_coords : bool
        Se True, retorna também as coordenadas dos centros

    Returns
    -------
    means : list
        Lista com a média ao redor de cada grão
    coords : array (opcional)
        Coordenadas dos centros detectados
    """

    # --- 1. Detectar centros (peaks locais) ---
    coords = peak_local_max(
        image,
        min_distance=min_distance,
        threshold_rel=threshold_rel,
        num_peaks=num_peaks
    )

    means = []

    # --- 2. Criar máscara circular ---
    y, x = np.ogrid[-radius:radius+1, -radius:radius+1]
    mask = x**2 + y**2 <= radius**2

    # --- 3. Iterar sobre centros ---
    for cy, cx in coords:
        # Definir bounding box
        y_min = max(cy - radius, 0)
        y_max = min(cy + radius + 1, image.shape[0])
        x_min = max(cx - radius, 0)
        x_max = min(cx + radius + 1, image.shape[1])

        patch = image[y_min:y_max, x_min:x_max]

        # Ajustar máscara se estiver na borda
        mask_crop = mask[
            (y_min - (cy - radius)):(y_max - (cy - radius)),
            (x_min - (cx - radius)):(x_max - (cx - radius))
        ]

        values = patch[mask_crop]

        if values.size > 0:
            means.append(values.mean())
        else:
            means.append(np.nan)

    if return_coords:
        return np.array(means), coords

    return np.array(means)

In [ ]:
def downsample_circle(image, centers, radius):
    """
    Calcula a média dos pixels dentro de círculos centrados em 'centers'.

    Parameters
    ----------
    image : 2D numpy array
        Imagem grayscale
    centers : array-like of shape (N, 2)
        Coordenadas dos centros (y, x)
    radius : int
        Raio do círculo em pixels

    Returns
    -------
    means : numpy array
        Média dos pixels dentro de cada círculo
    """

    means = []

    # máscara circular base
    y, x = np.ogrid[-radius:radius+1, -radius:radius+1]
    mask = x**2 + y**2 <= radius**2

    H, W = image.shape

    for cy, cx in centers:
        cy, cx = int(cy), int(cx)

        # bounding box
        y_min = max(cy - radius, 0)
        y_max = min(cy + radius + 1, H)
        x_min = max(cx - radius, 0)
        x_max = min(cx + radius + 1, W)

        patch = image[y_min:y_max, x_min:x_max]

        # ajuste da máscara (caso esteja na borda)
        mask_crop = mask[
            (y_min - (cy - radius)):(y_max - (cy - radius)),
            (x_min - (cx - radius)):(x_max - (cx - radius))
        ]

        values = patch[mask_crop]

        if values.size > 0:
            means.append(values.mean())
        else:
            means.append(np.nan)

    return np.array(means)

# Encoding
---

## Basket

In [ ]:
N = 50
input_vec = np.linspace(0, 0.3, N**2).reshape(-1, 1)
matrix_dist = distance_matrix(input_vec, input_vec)

fig, axs = plt.subplots(1, 2)

axs[0].imshow(matrix_dist, cmap='viridis')
axs[0].set_title("Distance Matrix (real values)")
axs[0].axis('off')

vec_basket = vectorized_basket(input_vec, 16, min_val=-0.15, max_val=0.3+0.15)
A = np.expand_dims(vec_basket, axis=1)  # (N, 1, nbin)
B = np.expand_dims(vec_basket, axis=0)  # (1, N, nbin)
matrix_basket_dist = np.abs(A - B).sum(axis=2)  # (N, N) -> Compute all pairwise L1 distances

axs[1].imshow(matrix_basket_dist, cmap='viridis')
axs[1].set_title("Distance Matrix (basket binarized)")
axs[1].axis('off')

plt.tight_layout(pad=.1)
plt.show()

In [ ]:
(matrix_basket_dist/matrix_basket_dist.max() - matrix_dist/matrix_dist.max()).sum() / matrix_basket_dist.size

In [ ]:
# normalize matrices and compute cosine similarity treating them as vectors
mb = matrix_basket_dist / matrix_basket_dist.max()
md = matrix_dist / matrix_dist.max()

v1 = mb.ravel()
v2 = md.ravel()

norm1 = np.linalg.norm(v1)
norm2 = np.linalg.norm(v2)

if norm1 == 0 or norm2 == 0:
    cos_sim = np.nan
else:
    cos_sim = np.dot(v1, v2) / (norm1 * norm2)

print("Cosine similarity:", cos_sim)

## Threshold

In [ ]:
data = np.load(r'C:\Users\lr699\Documents\dong-chaotic-systems-fundamentals\data\2026\run003\trial000.npz', allow_pickle=True)

In [ ]:
train_states = data['results'][0]['train_states']
time_vec = np.arange(train_states.shape[1])
neuron_idx = 0
neuron_signal = train_states[neuron_idx, :].flatten()

plt.figure()
plt.plot(time_vec[100:400], neuron_signal[100:400])
plt.ylabel(rf'Neuron signal $x(t)$')
plt.xlabel(rf'Time ($t$)')
plt.show()

In [ ]:
# Moving average with time

window = 500

neuron_idx = 10
neuron_signal = train_states[neuron_idx, :].flatten()

avg_signal = []
std_signal = []

for t in range(neuron_signal.shape[0] - window):
    avg_signal.append(neuron_signal[t:t+window+1].mean())
    std_signal.append(neuron_signal[t:t+window+1].std())

avg_signal, std_signal = np.array(avg_signal), np.array(std_signal)

fig, axs = plt.subplots(2, 1, sharex=True)


axs[0].plot(time_vec[window:], neuron_signal[window:])
axs[1].plot(time_vec[window:], avg_signal)

plt.show()

In [ ]:
inpt_dim = 1
res_dim = 512
window = 100

betas = train_states[:, -window:].mean(axis=1).reshape(-1, 1)
states = np.random.randn(res_dim).reshape(-1, 1)
inpt = np.random.rand(inpt_dim).reshape(-1, 1) * np.random.choice([-1, 1])

# States mask
states_activation = np.where(states >= betas, 1, 0) # Beta threshold

state_mask_dim = int(np.ceil(np.sqrt(res_dim)))
states_mask = np.zeros(state_mask_dim*state_mask_dim)
states_mask = states_mask.flatten()

for i, px in enumerate(states_activation.flatten()):
    states_mask[i] = px

states_mask = states_mask.reshape(state_mask_dim, state_mask_dim)

rep_h = DMD_HEIGHT // state_mask_dim
rep_w = DMD_WIDTH // state_mask_dim
rep = np.min((rep_h, rep_w))

states_mask = np.repeat(states_mask, repeats=rep, axis=0)
states_mask = np.repeat(states_mask, repeats=rep, axis=1)

# Input mask
inpt_mask = vectorized_basket(inpt, nbin=states_mask.shape[0], min_val=-1.5, max_val=1.5).T
inpt_mask = np.repeat(inpt_mask, repeats=states_mask.shape[0]//3, axis=1)

# Concat
dmd_mask = np.hstack((states_mask, inpt_mask))

plt.figure()
plt.imshow(dmd_mask)
plt.axis('off')

plt.show()

In [ ]:
res_dim = 512
inpt_dim = 1
window = 100

optical_setup = OpticalSetup(res_dim=res_dim)

betas = train_states[:res_dim, -window:].mean(axis=1).reshape(-1, 1)
states = np.random.randn(res_dim).reshape(-1, 1)
inpt = np.random.rand(inpt_dim).reshape(-1, 1) * np.random.choice([-1, 1])

dmd_mask = optical_setup.generate_dmd_mask2(state=states, beta=betas, inpt=inpt)

plt.figure()
plt.imshow(dmd_mask.squeeze())
plt.axis('off')

plt.show()

# Simulation
---

## a. "Perfect" mask

In [ ]:
temp_state = np.ones(5, dtype=np.float32).reshape(-1, 1) * 0.5
temp_inpt = np.ones(1, dtype=np.float32).reshape(-1, 1) * 0.5

vec_temp_state = vectorized_basket(array_inpt=temp_state, nbin=2**4, min_val=-0.5, max_val=1.5).reshape(1, -1)

plt.imshow(vec_temp_state)
plt.xticks([])
plt.yticks([])
plt.show()

vec_temp_inpt = vectorized_basket(array_inpt=temp_inpt, nbin=2*np.count_nonzero(vec_temp_state), min_val=-0.5, max_val=1.5)

plt.imshow(vec_temp_inpt)
plt.xticks([])
plt.yticks([])
plt.show()

np.count_nonzero(vec_temp_state), np.count_nonzero(vec_temp_inpt), np.count_nonzero(vec_temp_inpt)/np.count_nonzero(vec_temp_state) # -> Ratio 1/1 of mirrors turned on

In [ ]:
perfect_mask = np.hstack((vec_temp_state, vec_temp_inpt))

plt.imshow(perfect_mask)
plt.xticks([])
plt.yticks([])
plt.show()

## b. Optical setup

In [ ]:
res_dim = 512
state_nbin = int(2**3)

optical_setup = OpticalSetupSim(res_dim=res_dim, state_nbin=state_nbin, inpt_portion=2.0,
                                device='cpu', seed=7,
                                load_TMs=r'/home/lrvnc/Documents/research/dong-chaotic-systems-fundamentals/data/TMs512nbin8.npz')

plt.imshow(optical_setup._dummy_mask().T)
plt.xticks([])
plt.yticks([])
plt.show()

optical_setup._init_TM()
optical_setup._adjust_exposure(linear='auto', nonlinear='auto')

In [ ]:
# np.savez('TMs512nbin8.npz', TM_cam=optical_setup.TM_cam, TM_dmd=optical_setup.TM_dmd)

In [ ]:
(optical_setup.TM_cam @ optical_setup._dummy_mask()).shape

In [ ]:
(optical_setup.TM_dmd @ optical_setup._dummy_mask()).shape

In [ ]:
state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.random.rand(1).reshape(-1, 1)

plt.imshow(optical_setup.generate_dmd_mask(state=state, inpt=inpt).T)
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.random.rand(1).reshape(-1, 1)

speckle_lin, speckle_nonlin = optical_setup.compute_f(state=state, inpt=inpt, optical_features='both')

In [ ]:
optical_setup.reset_speckle_mem()

In [ ]:
state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.array(1.0).reshape(-1, 1)

# checkerboard_mask = optical_setup.get_checkerboard_mask()

fig, ax = plt.subplots(1, 2)

for idx, b in enumerate([None, np.random.rand(1).reshape(-1, 1)]):
    dmd_mask = optical_setup.generate_dmd_mask(state=state, inpt=inpt)

    ax[idx].imshow(dmd_mask.T)
    ax[idx].grid(False)
    ax[idx].set_xticks([])
    ax[idx].set_yticks([])

ax[0].set_title(f'Mask wo/ bias')
ax[1].set_title(f'Mask w/ bias')

plt.tight_layout()
plt.show()

In [ ]:
optical_setup._adjust_exposure(linear=1.0, nonlinear=1.0)

In [ ]:
state = np.random.rand(res_dim, 1).astype(np.float32)
inpt = np.random.rand(1, 1).astype(np.float32)
bias = np.ones((1, 1), dtype=np.float32) * 0.5

dmd_mask = optical_setup.generate_dmd_mask(state=state, inpt=inpt)
speckle_lin, speckle_nonlin = optical_setup.compute_f(state=state, inpt=inpt)

speckle_cam = speckle_lin

fig, ax = plt.subplots(1, 3, figsize=(12,4))

ax[0].imshow(reshape_to_square(dmd_mask.T))
ax[0].grid(False)
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_title(f'Mask')

im = ax[1].imshow(reshape_to_square(speckle_cam.T), cmap='inferno', vmin=0, vmax=255)
ax[1].grid(False)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].set_title(f'Speckle (Max: {speckle_cam.max()}, Min: {speckle_cam.min()})')
cbar = fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04, orientation='horizontal')
cbar.set_label('Intensity', labelpad=5)

ax[2].hist(speckle_cam.flatten(), bins=50, range=(0, 255), edgecolor='black')
ax[2].axvline(speckle_cam.flatten().mean(), label=f'Mean: {speckle_cam.flatten().mean()}')
ax[2].grid(True)
ax[2].set_title(f'Histogram')
ax[2].legend()

plt.tight_layout()
plt.show()

plt.plot(np.sort(speckle_lin.flatten()))
plt.plot(np.sort(speckle_nonlin.flatten()))

plt.show()

## c. Linearity of speckle

In [ ]:
res_dim = 4
nbin = 8
seed = 42

rng = np.random.default_rng(seed)

temp_state = np.ones(5, dtype=np.float32).reshape(-1, 1) * 0.5
temp_inpt = np.ones(1, dtype=np.float32).reshape(-1, 1) * 0.5

vec_temp_state = vectorized_basket(array_inpt=temp_state, nbin=nbin, min_val=-0.5, max_val=1.5).reshape(1, -1)
vec_temp_inpt = vectorized_basket(array_inpt=temp_inpt, nbin=2*nbin, min_val=-0.5, max_val=1.5)

concat_mask = np.hstack((vec_temp_state, vec_temp_inpt))

mask_dim = concat_mask.size

plt.imshow(concat_mask)
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
# single pass
tm_real = rng.standard_normal((res_dim, mask_dim), dtype=np.float32) # -> cam_dim = res_dim in our simplification
tm_imag = rng.standard_normal((res_dim, mask_dim), dtype=np.float32)

TM_concat = (tm_real + 1j*tm_imag).astype(np.complex64)
del tm_real, tm_imag

plt.imshow(np.abs(TM_concat))
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
TM_states = TM_concat[:, :vec_temp_state.flatten().size]
TM_inpt = TM_concat[:, vec_temp_state.flatten().size:vec_temp_state.flatten().size+vec_temp_inpt.flatten().size]

(TM_concat == np.hstack((TM_states, TM_inpt))).all()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(5,4))

axs[0].imshow(np.abs(TM_states))
axs[1].imshow(np.abs(TM_inpt))

for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
np.abs(TM_states @ vec_temp_state.T + TM_inpt @ vec_temp_inpt.T)

In [ ]:
np.abs(TM_concat @ concat_mask.T)

How I was imagining, it is linear (concatenating is just a way to vectorize everything).

# Real optical setup
---

In [ ]:
res_dim = 512
state_nbin = 16
optical_setup = OpticalSetup(monitoring=True, grid_points=1024, state_nbin=state_nbin, res_dim=res_dim)

In [ ]:
state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.random.rand(1).reshape(-1, 1)

dmd_mask = optical_setup.generate_dmd_mask(state, inpt, verbose=True)

plt.figure()
plt.imshow(dmd_mask.squeeze(), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
optical_setup._off()

In [ ]:
optical_setup._refresh_ref_speckle()

In [ ]:
speckle_image = optical_setup.ref_speckle_double
speckle_image = speckle_image / speckle_image.max()
speckle_image -= speckle_image.mean()
speckle_image = crop_center(speckle_image, 256, 256)

autocorr_2d_scipy = correlate2d(speckle_image, speckle_image, mode='full')
autocorr_2d_scipy /= autocorr_2d_scipy.max()

In [ ]:
fig, axs = plt.subplots(1,4, figsize=(12,3))
axs = axs.flatten()

axs[0].imshow(speckle_image, cmap='grey')
axs[0].set_title('Speckle')

axs[1].imshow(crop_center(autocorr_2d_scipy, 50, 50), cmap='magma')
axs[1].set_title('Autocorrelation SCPy')

autocorr_2d_scipy
i_max, j_max = np.unravel_index(autocorr_2d_scipy.argmax(), autocorr_2d_scipy.shape)
horizontal = autocorr_2d_scipy[i_max, :]
vertical = autocorr_2d_scipy[:, j_max]

rows, cols = autocorr_2d_scipy.shape
hshifts = np.arange(-cols//2, cols//2)
vshifts = np.arange(-rows//2, rows//2)

fwhm_x = compute_fwhm(hshifts, horizontal)
fwhm_y = compute_fwhm(vshifts, vertical)

axs[2].plot(hshifts, autocorr_2d_scipy[i_max, :])
axs[2].set_title('Horizontal slice on max')
axs[2].set_ylabel('Autocorrelation')
axs[2].set_xlabel('Pixel shift')
axs[2].grid(True)
axs[2].axhline(y=0.5, color='r', linestyle='--')
axs[2].text(15, 0.53, f'FWHM = {fwhm_x:.2f}', color='red', ha='left', fontsize=7)
axs[2].set_xlim([-50, 50])

axs[3].plot(vshifts, autocorr_2d_scipy[:, j_max])
axs[3].set_title('Vertical slice on max')
axs[3].set_ylabel('Autocorrelation')
axs[3].set_xlabel('Pixel shift')
axs[3].grid(True)
axs[3].axhline(y=0.5, color='r', linestyle='--')
axs[3].text(15, 0.53, f'FWHM = {fwhm_y:.2f}', color='red', ha='left', fontsize=7)
axs[3].set_xlim([-50, 50])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(speckle_image, cmap='gray')

radius = 3
threshold = 0.1

means, coords = speckle_grain_mean(
    speckle_image,
    radius=radius,
    min_distance=radius*2,
    threshold_rel=threshold,
    num_peaks=512,
    return_coords=True
)

for cy, cx in coords:
    circ = Circle((cx, cy), radius, edgecolor='red', facecolor='none', linewidth=1.5)
    ax.add_patch(circ)
    ax.plot(cx, cy, 'bo', markersize=1)

ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
means.shape

# LightPipes
---

## a. Learning

In [ ]:
# 1. Background
res_dim = 512
grid_size = 512
n_squares = 8
state_nbin = 8

state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.ones(1).reshape(-1, 1) * 0.5

rows = np.floor_divide(np.arange(grid_size), grid_size // n_squares, dtype=np.float32).reshape(-1, 1)
cols = np.floor_divide(np.arange(grid_size), grid_size // n_squares, dtype=np.float32).reshape(1, -1)
dmd_mask = (rows + cols) % 2

plt.imshow(dmd_mask)
plt.axis('off')
plt.show()

In [ ]:
state_bin = vectorized_basket(array_inpt=state, nbin=state_nbin, min_val=-0.5, max_val=1.5, clip=False)
state_bin = state_bin.reshape(2**5, -1) # -> controls state macropixel size
state_bin = np.repeat(state_bin, axis=1, repeats=grid_size // state_bin.shape[1])

inpt_bin = vectorized_basket(array_inpt=inpt.reshape(1, -1), nbin=grid_size, min_val=-0.5, max_val=1.5, clip=False)
rep = np.count_nonzero(state_bin) // np.count_nonzero(inpt_bin) // 2 # -> change state/input ratio
inpt_bin = np.repeat(inpt_bin, repeats=rep, axis=0)

inner_mask = np.vstack((state_bin, inpt_bin))
inner_mask = np.repeat(inner_mask, axis=0, repeats=grid_size//inner_mask.shape[0])

plt.imshow(inner_mask)
plt.axis('off')
plt.show()

In [ ]:
inner_mask.shape

In [ ]:
dmd_mask[:inner_mask.shape[0], :] = inner_mask
dmd_mask = np.roll(dmd_mask, (dmd_mask.shape[0] - inner_mask.shape[0]) // 2, axis=0)

In [ ]:
plt.imshow(dmd_mask)
plt.axis('off')
plt.show()

In [ ]:
rows = np.floor_divide(np.arange(grid_size), grid_size // n_squares, dtype=np.float32).reshape(-1, 1)
cols = np.floor_divide(np.arange(grid_size), grid_size // n_squares, dtype=np.float32).reshape(1, -1)
dmd_mask = (rows + cols) % 2

state_bin = vectorized_basket(array_inpt=state, nbin=state_nbin, min_val=-0.5, max_val=1.5, clip=False)
state_bin = state_bin.reshape(2**5, -1) # -> controls state macropixel size
state_bin = np.repeat(state_bin, axis=1, repeats=grid_size // state_bin.shape[1])

inpt_bin = vectorized_basket(array_inpt=inpt.reshape(1, -1), nbin=grid_size, min_val=-0.5, max_val=1.5, clip=False)
rep = np.count_nonzero(state_bin) // np.count_nonzero(inpt_bin) // 2 # -> change state/input ratio
inpt_bin = np.repeat(inpt_bin, repeats=rep, axis=0)

inner_mask = np.vstack((state_bin, inpt_bin))
inner_mask = np.repeat(inner_mask, axis=0, repeats=grid_size//inner_mask.shape[0])

dmd_mask[:inner_mask.shape[0], :] = inner_mask
dmd_mask = np.roll(dmd_mask, (dmd_mask.shape[0] - inner_mask.shape[0]) // 2, axis=0)

plt.imshow(dmd_mask)
plt.axis('off')
plt.show()

In [ ]:
rng = np.random.default_rng(42)
tm_real = rng.standard_normal((N, N), dtype=np.float32)
tm_imag = rng.standard_normal((N, N), dtype=np.float32)

tm = tm_real + 1j*tm_imag

complex_media_I = np.abs(tm)
complex_media_phase = np.angle(tm)

In [ ]:
res_dim = 512
grid_size = 512
n_squares = 8
state_nbin = 8

dmd_px_h = 1080
dmd_px_w = 1920
micromirror_pitch = 10.8*um

z_dmd1_diff  = 20*cm
z_diff_dmd2  = 20*cm
z_dmd2_cam   = 20*cm

wavelength = 632.8*nm # HeNe
grid_dim = 512 # grid points (N x N)
grid_size = micromirror_pitch*grid_dim # physical window of my simulation

print(f'DMD dimensions: {grid_size*100:.2f}cm x {grid_size*100:.2f}cm')

state = np.random.rand(res_dim).reshape(-1, 1)
inpt = np.ones(1).reshape(-1, 1) * 0.5

# DMD mask
rows = np.floor_divide(np.arange(grid_dim), grid_dim // n_squares, dtype=np.float32).reshape(-1, 1)
cols = np.floor_divide(np.arange(grid_dim), grid_dim // n_squares, dtype=np.float32).reshape(1, -1)
dmd_mask = (rows + cols) % 2

state_bin = vectorized_basket(array_inpt=state, nbin=state_nbin, min_val=-0.5, max_val=1.5, clip=False)
state_bin = state_bin.reshape(2**5, -1) # -> controls state macropixel size
state_bin = np.repeat(state_bin, axis=1, repeats=grid_dim // state_bin.shape[1])

inpt_bin = vectorized_basket(array_inpt=inpt.reshape(1, -1), nbin=grid_dim, min_val=-0.5, max_val=1.5, clip=False)
rep = np.count_nonzero(state_bin) // np.count_nonzero(inpt_bin) // 2 # -> change state/input ratio
inpt_bin = np.repeat(inpt_bin, repeats=rep, axis=0)

inner_mask = np.vstack((state_bin, inpt_bin))
inner_mask = np.repeat(inner_mask, axis=0, repeats=grid_dim//inner_mask.shape[0])

dmd_mask[:inner_mask.shape[0], :] = inner_mask
dmd_mask = np.roll(dmd_mask, (dmd_mask.shape[0] - inner_mask.shape[0]) // 2, axis=0)

# DMD first hit
beam = Begin(labda=wavelength, size=grid_size, N=grid_dim, dtype=np.complex64)
beam = MultIntensity(beam, dmd_mask)
beam = Fresnel(beam, z_dmd1_diff, usepyFFTW=True)

# Diffuser (complex media)
beam = RandomPhase(beam, maxPhase=2*np.pi)
# beam = Fresnel(beam, z_diff_dmd2)

# DMD mask again
# beam = MultIntensity(beam, dmd_mask)
beam = Fresnel(beam, z_dmd2_cam, usepyFFTW=True)

# Cam
speckle = Intensity(beam)
# speckle -= speckle.mean()
speckle /= speckle.max()

# (Optional) visualize
fig, axs = plt.subplots(1, 3, layout='constrained', figsize=(10,3))

axs[0].imshow(dmd_mask)
axs[0].axis('off')

img = axs[1].imshow(speckle, cmap="gray")
# img.colorbar(label="Intensity (a.u.)")
axs[1].set_title("Camera plane intensity")
axs[1].axis("off")

axs[2].hist(speckle.flatten())

# plt.tight_layout()
plt.show()

In [ ]:
speckle.min()

In [ ]:
def compute_fwhm(x, y):
    y = y / np.max(y)
    half_max = 0.5

    above = y >= half_max
    if not np.any(above):
        return None

    indices = np.where(above)[0]
    left_idx = indices[0] - 1
    right_idx = indices[-1] + 1

    # Interpolation
    f = interp1d(x, y, kind='linear', bounds_error=False, fill_value='extrapolate')
    x_fine = np.linspace(x[left_idx], x[right_idx], 1000)
    y_fine = f(x_fine)

    cross_points = x_fine[np.where(np.diff(np.sign(y_fine - half_max)) != 0)]

    if len(cross_points) >= 2:
        return np.abs(cross_points[-1] - cross_points[0])
    else:
        return None

# speckle -= speckle.mean()
autocorr_2d_scipy = correlate2d(speckle[256-64:256+64,256-64:256+64], speckle[256-64:256+64,256-64:256+64], mode='full')
autocorr_2d_scipy /= autocorr_2d_scipy.max()

fig, axs = plt.subplots(1,4, figsize=(12,3))
axs = axs.flatten()

axs[0].imshow(speckle, cmap='grey')
axs[0].set_title('Speckle')

axs[1].imshow(autocorr_2d_scipy, cmap='magma')
axs[1].set_title('Autocorrelation SCPy')

autocorr_2d_scipy
i_max, j_max = np.unravel_index(autocorr_2d_scipy.argmax(), autocorr_2d_scipy.shape)
horizontal = autocorr_2d_scipy[i_max, :]
vertical = autocorr_2d_scipy[:, j_max]

rows, cols = autocorr_2d_scipy.shape
hshifts = np.arange(-cols//2, cols//2)
vshifts = np.arange(-rows//2, rows//2)

fwhm_x = compute_fwhm(hshifts, horizontal)
fwhm_y = compute_fwhm(vshifts, vertical)

axs[2].plot(hshifts, autocorr_2d_scipy[i_max, :])
axs[2].set_title('Horizontal slice on max')
axs[2].set_ylabel('Autocorrelation')
axs[2].set_xlabel('Pixel shift')
axs[2].grid(True)
axs[2].axhline(y=0.5, color='r', linestyle='--')
axs[2].text(15, 0.53, f'FWHM = {fwhm_x:.2f}', color='red', ha='left', fontsize=7)
axs[2].set_xlim([-50, 50])

axs[3].plot(vshifts, autocorr_2d_scipy[:, j_max])
axs[3].set_title('Vertical slice on max')
axs[3].set_ylabel('Autocorrelation')
axs[3].set_xlabel('Pixel shift')
axs[3].grid(True)
axs[3].axhline(y=0.5, color='r', linestyle='--')
axs[3].text(15, 0.53, f'FWHM = {fwhm_y:.2f}', color='red', ha='left', fontsize=7)
axs[3].set_xlim([-50, 50])

plt.tight_layout()
plt.show()

In [ ]:
grid_size = 30*mm
grid_dim = 5
lambda_ = 632*nm
beam = Begin(size=grid_size, labda=lambda_, N=grid_dim, dtype=np.complex64)
beam = RandomPhase(Fin=beam, seed=42, maxPhase=2*np.pi) # -> Always the same after Begin

beam.field

In [ ]:
# Grid and wavelength (HeNe)
wavelength = 632.8*nm
size = 2*cm # physical size of simulation window
N = 1920 # grid points (N x N)

# Distances between elements
z_dmd1_diff  = 10*cm
z_diff_dmd2  = 10*cm
z_dmd2_cam   = 10*cm

# Start field: plane or Gaussian beam
F = Begin(size, wavelength, N)
# F = GaussBeam(F, w0=2*mm)   # "Laser HeNe"

# --- DMD 1 (amplitude mask 0/1) ---
dmd1 = np.random.choice([0.0, 1.0], size=(N, N))
F = MultIntensity(F, dmd1)

# --- Propagate to diffuser ---
F = Fresnel(F, z_dmd1_diff, usepyFFTW=True)

# --- Diffuser (random phase screen) ---
F = RandomPhase(F, maxPhase=np.pi)

# --- Propagate to DMD 2 ---
F = Fresnel(F, z_diff_dmd2)

# --- DMD 2 (another amplitude mask) ---
# dmd2 = np.random.choice([0,1], size=(N, N))
# F = MultIntensity(F, dmd2)

# # --- Propagate to camera ---
# F = Fresnel(F, z_dmd2_cam)

# Camera intensity (speckle pattern)
I = Intensity(F)

# (Optional) visualize
plt.figure()
plt.imshow(I, cmap="gray")
plt.colorbar(label="Intensity (a.u.)")
plt.title("Camera plane intensity")
plt.axis("off")
plt.show()

In [ ]:
F = Begin(size, wavelength, N)
F = PlaneWave(F, w=2*cm)

fig, axs = plt.subplots(1, 2)

axs[0].imshow(np.abs(F.field))
axs[0].set_title('Magnitude')

axs[1].imshow(np.angle(F.field))
axs[1].set_title('Phase')

for ax in axs:
    ax.axis('off')

plt.show()

In [ ]:
F = Begin(size, wavelength, N)
# F = GaussBeam(F, w0=2*mm)   # "Laser HeNe"
dmd_mask = np.ones((N, N))
dmd_mask[:N//2, :] = 0
F = MultIntensity(F, dmd_mask) # -> Element wise multiplication: our DMD (0: OFF, 1: ON)

fig, axs = plt.subplots(1, 2)

axs[0].imshow(np.abs(F.field))
axs[0].set_title('Magnitude')

axs[1].imshow(np.angle(F.field))
axs[1].set_title('Phase')

for ax in axs:
    ax.axis('off')

plt.show()

In [ ]:
rng = np.random.default_rng(42)
tm_real = rng.standard_normal((N, N), dtype=np.float32)
tm_imag = rng.standard_normal((N, N), dtype=np.float32)

tm = tm_real + 1j*tm_imag

complex_media_I = np.abs(tm)
complex_media_phase = np.angle(tm)

In [ ]:
plt.hist(complex_media_phase.flatten())

In [ ]:
F = Begin(size, wavelength, N)

# Simulating the complex media
F = MultIntensity(F, Intens=complex_media_I)
F = MultPhase(F, Phi=complex_media_phase)
F = Normal(F)

fig, axs = plt.subplots(1, 2)

axs[0].imshow(np.abs(F.field))
axs[0].set_title('Magnitude')

axs[1].imshow(np.angle(F.field))
axs[1].set_title('Phase')

for ax in axs:
    ax.axis('off')

plt.show()

In [ ]:
plt.hist(np.angle(F.field).flatten())

## b. Setup

In [ ]:
res_dim = 512
distances = {
    'z_dmd_diff':  20*cm,
    'z_diff_cam_lin':  35*cm,
    'z_diff_dmd_nonlin':  20*cm,
    'z_dmd_cam_nonlin':  20*cm,
}

optical_setup = OpticalSetupSim2(res_dim=res_dim, **distances)
optical_setup._adjust_exposure(linear=None, nonlinear=None)

In [ ]:
state = np.random.rand(res_dim).reshape(-1, 1)
# state = np.ones(res_dim).reshape(-1, 1) * 0.5
inpt = np.ones(1).reshape(-1, 1) * 0.5

dmd_mask = optical_setup.generate_dmd_mask(state=state, inpt=inpt)
speckle_lin, speckle_nonlin = optical_setup.compute_f(state=state, inpt=inpt, simulate_cam=True, downsample=False)

fig, axs = plt.subplots(1, 3)

axs[0].imshow(dmd_mask, cmap='gray', vmin=0, vmax=1)
axs[0].set_title('DMD mask')
axs[1].imshow(speckle_lin, cmap='gray')
axs[1].set_title('Single pass')
axs[2].imshow(speckle_nonlin, cmap='gray')
axs[2].set_title('Double pass')

for ax in axs:
    ax.axis('off')

plt.show()

In [ ]:
def compute_fwhm(x, y):
    y = y / np.max(y)
    half_max = 0.5

    above = y >= half_max
    if not np.any(above):
        return None

    indices = np.where(above)[0]
    left_idx = indices[0] - 1
    right_idx = indices[-1] + 1

    # Interpolation
    f = interp1d(x, y, kind='linear', bounds_error=False, fill_value='extrapolate')
    x_fine = np.linspace(x[left_idx], x[right_idx], 1000)
    y_fine = f(x_fine)

    cross_points = x_fine[np.where(np.diff(np.sign(y_fine - half_max)) != 0)]

    if len(cross_points) >= 2:
        return np.abs(cross_points[-1] - cross_points[0])
    else:
        return None

speckle = speckle_lin

speckle -= speckle.mean()
autocorr_2d_scipy = correlate2d(speckle[256-64:256+64,256-64:256+64], speckle[256-64:256+64,256-64:256+64], mode='full')
autocorr_2d_scipy /= autocorr_2d_scipy.max()

fig, axs = plt.subplots(1,4, figsize=(12,3))
axs = axs.flatten()

axs[0].imshow(speckle, cmap='grey')
axs[0].set_title('Speckle')
axs[0].axis('off')

axs[1].imshow(autocorr_2d_scipy, cmap='magma')
axs[1].set_title('Autocorrelation SCPy')
axs[1].axis('off')

autocorr_2d_scipy
i_max, j_max = np.unravel_index(autocorr_2d_scipy.argmax(), autocorr_2d_scipy.shape)
horizontal = autocorr_2d_scipy[i_max, :]
vertical = autocorr_2d_scipy[:, j_max]

rows, cols = autocorr_2d_scipy.shape
hshifts = np.arange(-cols//2, cols//2)
vshifts = np.arange(-rows//2, rows//2)

fwhm_x = compute_fwhm(hshifts, horizontal)
fwhm_y = compute_fwhm(vshifts, vertical)

axs[2].plot(hshifts, autocorr_2d_scipy[i_max, :])
axs[2].set_title('Horizontal slice on max')
axs[2].set_ylabel('Autocorrelation')
axs[2].set_xlabel('Pixel shift')
axs[2].grid(True)
axs[2].axhline(y=0.5, color='r', linestyle='--')
axs[2].text(15, 0.53, f'FWHM = {fwhm_x:.2f}', color='red', ha='left', fontsize=7)
axs[2].set_xlim([-50, 50])

axs[3].plot(vshifts, autocorr_2d_scipy[:, j_max])
axs[3].set_title('Vertical slice on max')
axs[3].set_ylabel('Autocorrelation')
axs[3].set_xlabel('Pixel shift')
axs[3].grid(True)
axs[3].axhline(y=0.5, color='r', linestyle='--')
axs[3].text(15, 0.53, f'FWHM = {fwhm_y:.2f}', color='red', ha='left', fontsize=7)
axs[3].set_xlim([-50, 50])

plt.tight_layout()
plt.show()

In [ ]:
def circular_downsample(img: np.ndarray, s: int, radius: int | None = None, agg: str = "mean") -> np.ndarray:
    """
    Downsample an image by sampling circular regions on a regular grid.

    Parameters
    ----------
    img : np.ndarray
        Input image, shape (H, W) or (H, W, C).
    s : int
        Grid spacing (in pixels). Each grid cell is s×s.
    radius : int or None
        Radius (in pixels) of the circle inside each s×s cell.
        If None, defaults to s // 2.
    agg : {"mean", "median"}
        Aggregation to apply inside each circle.

    Returns
    -------
    np.ndarray
        Downsampled image of shape (H // s, W // s) or (H // s, W // s, C).
    """
    if s <= 0:
        raise ValueError("s must be a positive integer.")

    if img.ndim == 2:
        H, W = img.shape
        C = None
    elif img.ndim == 3:
        H, W, C = img.shape
    else:
        raise ValueError("img must be 2D (H, W) or 3D (H, W, C).")

    radius = radius if radius is not None else s // 2

    # Number of grid cells fully fitting inside the image
    n_rows = H // s
    n_cols = W // s

    if n_rows == 0 or n_cols == 0:
        raise ValueError("Grid spacing s is larger than the image dimensions.")

    # Precompute circular mask inside an s×s window
    yy, xx = np.ogrid[:s, :s]
    cy, cx = s // 2, s // 2  # center of the window
    circle_mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2

    circle_mask = np.tile(circle_mask, reps=(n_rows, n_cols))
    downsampled_img = img[circle_mask]

    pixels_per_cell = circle_mask[:s, :s].sum()

    # Reshape into (n_cells, pixels_per_cell)
    downsampled_img = downsampled_img.reshape(-1, pixels_per_cell)

    # Apply aggregation
    if agg is None:
        pass
    elif agg == "mean":
        downsampled_img = downsampled_img.mean(axis=1)
    elif agg == "median":
        downsampled_img = np.median(downsampled_img, axis=1)
    else:
        raise ValueError("agg must be None, 'mean', or 'median'.")

    # Reshape
    if agg is None:
        return downsampled_img, circle_mask # each row one speckle grain
    else:
        downsampled_img = downsampled_img.reshape(n_rows, n_cols) # each pixel one speckle grain
        return downsampled_img, circle_mask

In [ ]:
downsampled_speckle, circle_mask = circular_downsample(img=speckle_lin, s=16, radius=3, agg='mean')

plt.figure()
plt.imshow(downsampled_speckle, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
downsampled_speckle.min(), downsampled_speckle.max()/255 * 9

In [ ]:
plt.figure()
plt.hist(downsampled_speckle.flatten(), edgecolor='black')
plt.tight_layout()
plt.show()